# ARC NeuroGolf static ONNX solver

Reference layout adapted from the uploaded fill/additive-marking notebook. The task-specific modelling cell uses a semantic feature-tree or a symbolic reflection builder, not raw output-template lookup.

In [1]:
!rm -rf /kaggle/working/*
%reset -f

In [2]:
COMPETITION = '/kaggle/input/competitions/neurogolf-2026'

In [3]:
import importlib.util, subprocess, sys
missing=[p for p in ['onnx','onnxruntime','onnxscript','torch','numpy'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
print('dependencies ok')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 6.9 MB/s eta 0:00:00
dependencies ok


In [4]:
import json, os, time, hashlib, zipfile,  csv, base64
import glob, sys,math, random, collections,io,shutil
from pathlib import Path
import numpy as np
import torch
import onnx
import onnxruntime as ort
import torch, torch.nn as nn, torch.nn.functional as F
from collections import defaultdict, Counter
from onnx import shape_inference

In [5]:
TASK_ID = "task375"
TASK_PATH = Path(COMPETITION)/"task375.json"
OUT_DIR = Path.cwd()/"task375_x_diagonal_onnx"
ONNX_PATH = OUT_DIR / f"{TASK_ID}.onnx"
SUMMARY_PATH = OUT_DIR / f"{TASK_ID}_validation_summary.json"
SUBMISSION_PATH = Path.cwd()/"submission.zip"


OUT_DIR.mkdir(parents=True, exist_ok=True)
FORBIDDEN_OPS = {"Loop", "Scan", "NonZero", "Unique", "Script", "Function"}
INPUT_SHAPE = [1, 10, 30, 30]
OUTPUT_SHAPE = [1, 10, 30, 30]

with TASK_PATH.open() as f:
    task = json.load(f)
print({k: len(v) for k, v in task.items()})

{'train': 3, 'test': 1, 'arc-gen': 54}


In [6]:
def encode_grid(grid):
    arr = np.zeros((1, 10, 30, 30), dtype=np.float32)
    h, w = len(grid), len(grid[0])
    assert h <= 30 and w <= 30
    for r, row in enumerate(grid):
        for c, value in enumerate(row):
            arr[0, int(value), r, c] = 1.0
    return arr


def validate_model(examples, run):
    ok = 0
    bad = []
    outside_ok = 0
    active_ok = 0
    for i, ex in enumerate(examples):
        x = encode_grid(ex["input"])
        y = run(x)
        exp = encode_grid(ex["output"])
        if np.array_equal(y, exp):
            ok += 1
        else:
            bad.append(i)
        h, w = len(ex["output"]), len(ex["output"][0])
        outside = y.copy()
        outside[:, :, :h, :w] = 0
        outside_ok += int(np.all(outside == 0))
        inside_sum = y[:, :, :h, :w].sum(axis=1)
        active_ok += int(np.all(inside_sum == 1.0))
    return {"ok": ok, "total": len(examples), "bad_first10": bad[:10], "outside_zero_ok": outside_ok, "active_canvas_covered_ok": active_ok}


In [7]:
class Task375XDiagonal(nn.Module):
    def __init__(self):
        super().__init__()
        rr = torch.arange(30, dtype=torch.float32).view(1, 30, 1).expand(1, 30, 30)
        cc = torch.arange(30, dtype=torch.float32).view(1, 1, 30).expand(1, 30, 30)
        row_ids = torch.arange(30, dtype=torch.float32).view(1, 30)
        col_ids = torch.arange(30, dtype=torch.float32).view(1, 30)
        zero_oh = torch.zeros(1, 10, 1, 1, dtype=torch.float32)
        zero_oh[:, 0:1] = 1.0
        self.register_buffer("rr", rr)
        self.register_buffer("cc", cc)
        self.register_buffer("row_ids", row_ids)
        self.register_buffer("col_ids", col_ids)
        self.register_buffer("zero_oh", zero_oh)

    def forward(self, x):
        active_f = (x.sum(dim=1) > 0.0).to(x.dtype)
        active = active_f > 0.5
        row_has = (active_f.max(dim=2).values > 0.5).to(x.dtype)
        col_has = (active_f.max(dim=1).values > 0.5).to(x.dtype)
        row_ids = self.row_ids.to(x.device)
        col_ids = self.col_ids.to(x.device)
        min_r = torch.min(row_ids * row_has + (1.0 - row_has) * 30.0, dim=1).values.view(1, 1, 1)
        max_r = torch.max(row_ids * row_has, dim=1).values.view(1, 1, 1)
        min_c = torch.min(col_ids * col_has + (1.0 - col_has) * 30.0, dim=1).values.view(1, 1, 1)
        max_c = torch.max(col_ids * col_has, dim=1).values.view(1, 1, 1)

        rr = self.rr.to(x.device)
        cc = self.cc.to(x.device)
        dr = rr - min_r
        dc = cc - min_c
        side = max_r - min_r
        diag1 = torch.abs(dr - dc) < 0.5
        diag2 = torch.abs((dr + dc) - side) < 0.5
        diag = active & (diag1 | diag2)
        diag_f = diag.unsqueeze(1).to(x.dtype)
        out = x * (1.0 - diag_f) + self.zero_oh.to(x.device) * diag_f
        return out * active.unsqueeze(1).to(x.dtype)

model = Task375XDiagonal().eval()


In [8]:
with torch.no_grad():
    for split in ["train", "test", "arc-gen"]:
        stats = validate_model(task[split], lambda z: model(torch.from_numpy(z)).numpy())
        print(split, stats)
        assert stats["ok"] == stats["total"]


train {'ok': 3, 'total': 3, 'bad_first10': [], 'outside_zero_ok': 3, 'active_canvas_covered_ok': 3}
test {'ok': 1, 'total': 1, 'bad_first10': [], 'outside_zero_ok': 1, 'active_canvas_covered_ok': 1}
arc-gen {'ok': 54, 'total': 54, 'bad_first10': [], 'outside_zero_ok': 54, 'active_canvas_covered_ok': 54}


In [9]:
dummy = torch.from_numpy(encode_grid(task["test"][0]["input"]))
torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    input_names=["input"],
    output_names=["output"],
    opset_version=17,
    do_constant_folding=True,
    dynamic_axes=None,
    dynamo=False,
    external_data=False,
)
onnx_model = onnx.load(str(ONNX_PATH))
onnx.checker.check_model(onnx_model)
ops = Counter(node.op_type for node in onnx_model.graph.node)
forbidden = sorted(set(ops) & FORBIDDEN_OPS)
print("ONNX size", ONNX_PATH.stat().st_size)
print("Ops", ops)
print("Forbidden", forbidden, "functions", len(onnx_model.functions))
assert ONNX_PATH.stat().st_size < 1_400_000
assert not forbidden
assert len(onnx_model.functions) == 0


/tmp/ipykernel_16/659460508.py:2: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


ONNX size 12222
Ops Counter({'Constant': 17, 'Sub': 8, 'Mul': 7, 'Cast': 5, 'Greater': 4, 'Add': 4, 'ReduceMax': 3, 'Reshape': 3, 'ReduceMin': 2, 'Abs': 2, 'Less': 2, 'Unsqueeze': 2, 'Identity': 1, 'ReduceSum': 1, 'Or': 1, 'And': 1})
Forbidden [] functions 0


In [10]:
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
assert list(sess.get_inputs()[0].shape) == INPUT_SHAPE, sess.get_inputs()[0].shape
assert list(sess.get_outputs()[0].shape) == OUTPUT_SHAPE, sess.get_outputs()[0].shape

def run_onnx(z):
    return sess.run(None, {"input": z})[0]

holdout = task["arc-gen"][: int(np.ceil(0.6 * len(task["arc-gen"])))]
onnx_stats = {
    "train": validate_model(task["train"], run_onnx),
    "test": validate_model(task["test"], run_onnx),
    "arc_gen_holdout_60pct": validate_model(holdout, run_onnx),
    "arc_gen_all": validate_model(task["arc-gen"], run_onnx),
}
print(json.dumps(onnx_stats, indent=2))
assert onnx_stats["train"]["ok"] == onnx_stats["train"]["total"]
assert onnx_stats["test"]["ok"] == onnx_stats["test"]["total"]
assert onnx_stats["arc_gen_all"]["ok"] == onnx_stats["arc_gen_all"]["total"]


{
  "train": {
    "ok": 3,
    "total": 3,
    "bad_first10": [],
    "outside_zero_ok": 3,
    "active_canvas_covered_ok": 3
  },
  "test": {
    "ok": 1,
    "total": 1,
    "bad_first10": [],
    "outside_zero_ok": 1,
    "active_canvas_covered_ok": 1
  },
  "arc_gen_holdout_60pct": {
    "ok": 33,
    "total": 33,
    "bad_first10": [],
    "outside_zero_ok": 33,
    "active_canvas_covered_ok": 33
  },
  "arc_gen_all": {
    "ok": 54,
    "total": 54,
    "bad_first10": [],
    "outside_zero_ok": 54,
    "active_canvas_covered_ok": 54
  }
}


In [11]:
summary = {
    "task_id": TASK_ID,
    "model": "x_diagonal_coordinate_projection",
    "input_shape": INPUT_SHAPE,
    "output_shape": OUTPUT_SHAPE,
    "onnx_path": str(ONNX_PATH),
    "onnx_size_bytes": ONNX_PATH.stat().st_size,
    "ops": dict(ops),
    "forbidden_ops": forbidden,
    "function_count": len(onnx_model.functions),
    "onnx_stats": onnx_stats,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

for path in [SUBMISSION_PATH]:
    if path.exists():
        path.unlink()
    with zipfile.ZipFile(path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(ONNX_PATH, arcname=f"{TASK_ID}.onnx")

print("summary", SUMMARY_PATH)
print("submission", SUBMISSION_PATH, SUBMISSION_PATH.stat().st_size)



summary /kaggle/working/task375_x_diagonal_onnx/task375_validation_summary.json
submission /kaggle/working/submission.zip 1517
